# Conversational alignment

Day 3: 320 reviewed matched pairs, ten per each of 32 emotions. Lexical overlap uses adjacent word-set Jaccard. Semantic similarity uses adjacent cosine similarity from pretrained all-MiniLM-L6-v2 embeddings. These are descriptive alignment measures, not proof of adaptation or empathy.

In [ ]:
from pathlib import Path
import hashlib
import json
import importlib.metadata
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() or (p / 'scripts/achraf').is_dir()), Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data/processed/achraf'
import re
OUTPUT_DIR = PROJECT_ROOT / 'results/achraf/day3'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_FILE = DATA_DIR / 'day3_reviewed_320_v2.csv'
MODEL_INPUT_FILE = DATA_DIR / 'model_inputs.csv'
RECOMPUTE_SEMANTIC = True
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EXPECTED_PAIRS = 320

def file_fingerprint(path):
    return hashlib.sha256(path.read_bytes().replace(b'\r\n', b'\n')).hexdigest()

METADATA_FILE = OUTPUT_DIR / 'alignment_run_metadata.json'
CACHE_FILE = OUTPUT_DIR / 'transition_scores.csv'
previous_metadata = json.loads(METADATA_FILE.read_text(encoding='utf-8')) if METADATA_FILE.is_file() else {}

def validate_cache():
    if previous_metadata.get('cache_input_sha256_lf') != file_fingerprint(INPUT_FILE) or previous_metadata.get('cache_sha256_lf') != file_fingerprint(CACHE_FILE):
        raise ValueError('Cache or input changed. Rerun model inference.')
    if previous_metadata.get('cache_model_inputs_sha256_lf') != file_fingerprint(MODEL_INPUT_FILE) or previous_metadata.get('cache_text_correction') != 'hit:9551_conv:19102: mitig ates -> mitigates':
        raise ValueError('Source data or correction changed. Rerun model inference.')


## 1. Validate the reviewed selection and its source rows

The correction `mitig ates` → `mitigates` applies only to the known GPT contribution in working memory. It never rewrites the original CSV. Stored model text must match `model_inputs.csv` after whitespace normalization. This check verifies provenance, not a new manual semantic review.

In [ ]:
df = pd.read_csv(INPUT_FILE)
required = {'pair_id', 'emotion', 'prompt', 'human_contributions', 'gpt_contributions', 'human_contribution_count', 'gpt_contribution_count', 'model_human_text', 'model_gpt_text', 'review_status', 'review_notes'}
if not required.issubset(df.columns) or len(df) != EXPECTED_PAIRS or df['pair_id'].duplicated().any() or df[list(required - {'review_notes'})].isna().any().any():
    raise ValueError('Invalid reviewed-pair schema, row count, IDs or missing fields')
if df.emotion.nunique() != 32 or not df.emotion.value_counts().eq(10).all():
    raise ValueError('Expected ten reviewed pairs per emotion')
model_inputs = pd.read_csv(MODEL_INPUT_FILE)
if not {'pair_id', 'source', 'text', 'context', 'prompt'}.issubset(model_inputs.columns) or model_inputs.duplicated(['pair_id', 'source']).any():
    raise ValueError('Invalid model-input schema or duplicate source keys')
model_index = model_inputs.set_index(['pair_id', 'source'])

def normalized_text(text):
    return re.sub('\\s+', ' ', str(text)).strip()
for _, row in df.iterrows():
    for source in ['human', 'gpt']:
        if (row.pair_id, source) not in model_index.index:
            raise ValueError(f'Missing source row: {row.pair_id}, {source}')
        original = model_index.loc[row.pair_id, source]
        if row.emotion != original.context or normalized_text(row.prompt) != normalized_text(str(original.prompt).replace('_comma_', ',')):
            raise ValueError(f'Source emotion/prompt mismatch: {row.pair_id}, {source}')
        if normalized_text(row[f'model_{source}_text']) != normalized_text(original.text):
            raise ValueError(f'Stored source text mismatch: {row.pair_id}, {source}')
affected = df['pair_id'].eq('hit:9551_conv:19102')
typo_count = int(df.loc[affected, 'gpt_contributions'].str.count('mitig ates').sum())
df.loc[affected, 'gpt_contributions'] = df.loc[affected, 'gpt_contributions'].str.replace('mitig ates', 'mitigates', regex=False)
print('Matched pairs:', len(df), 'Known typo replacements:', typo_count)


In [ ]:
def parse_contributions(contributions):
    turns = []
    for line in contributions.splitlines():
        match = re.fullmatch('(Human_\\d+|Speaker|Listener): (.+)', line)
        if match is None or not match.group(2).strip():
            raise ValueError(f'Invalid contribution line: {line!r}')
        turns.append((match.group(1), match.group(2)))
    if len(turns) < 2 or len({speaker for speaker, _ in turns}) != 2:
        raise ValueError('Expected at least two contributions and exactly two participants')
    if any((left[0] == right[0] for left, right in zip(turns, turns[1:]))):
        raise ValueError('Consecutive contributions have the same participant')
    return turns
for source in ['human', 'gpt']:
    for _, row in df.iterrows():
        turns = parse_contributions(row[f'{source}_contributions'])
        if len(turns) != row[f'{source}_contribution_count']:
            raise ValueError(f'Contribution-count mismatch: {row.pair_id}, {source}')
print('Dialogues checked:', len(df) * 2, 'Issues found: 0')


## 2. Adjacent lexical and semantic scores

Tokenization preserves the original analysis: lowercase English letter tokens with straight-apostrophe contractions. It includes function words. Empty unions are undefined and must not be replaced with zero. Semantic inference is duplicated for the dialogue/transition reconciliation, as in the original notebook.

Set `RECOMPUTE_SEMANTIC = False` only to reuse the matching supplied `transition_scores.csv`. Cache mode recomputes lexical scores and checks turn counts/positions, but does not rerun embeddings.

In [ ]:
def lexical_overlap(text_1, text_2):
    words_1 = set(re.findall("[a-z]+(?:'[a-z]+)*", text_1.lower()))
    words_2 = set(re.findall("[a-z]+(?:'[a-z]+)*", text_2.lower()))
    shared_words = words_1 & words_2
    all_words = words_1 | words_2
    if len(all_words) == 0:
        return None
    jaccard = len(shared_words) / len(all_words)
    return jaccard


In [ ]:
def conversation_overlap(contributions):
    lines = contributions.splitlines()
    turns = []
    for line in lines:
        participant, text = line.split(': ', 1)
        turns.append((participant, text))
    scores = []
    for i in range(1, len(turns)):
        text_1 = turns[i - 1][1]
        text_2 = turns[i][1]
        score = lexical_overlap(text_1, text_2)
        if score is not None:
            scores.append(score)
    if len(scores) == 0:
        return None
    return sum(scores) / len(scores)


In [ ]:
def conversation_semantic_similarity(contributions):
    texts = [line.split(': ', 1)[1] for line in contributions.splitlines()]
    if len(texts) < 2:
        return None
    embeddings = model.encode(texts)
    scores = []
    for i in range(1, len(embeddings)):
        score = model.similarity(embeddings[i - 1:i], embeddings[i:i + 1]).item()
        scores.append(score)
    return sum(scores) / len(scores)


In [ ]:
df['human_overlap'] = df['human_contributions'].apply(conversation_overlap)
df['gpt_overlap'] = df['gpt_contributions'].apply(conversation_overlap)
df['overlap_difference'] = df['gpt_overlap'] - df['human_overlap']
if df[['human_overlap', 'gpt_overlap']].isna().any().any():
    raise ValueError('Undefined lexical dialogue scores')
if RECOMPUTE_SEMANTIC:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME)
    print('Model:', MODEL_NAME, 'Device:', model.device)
    for source in ['human', 'gpt']:
        texts = [text for conversation in df[f'{source}_contributions'] for _, text in parse_contributions(conversation)]
        lengths = model.tokenizer(texts, truncation=False, padding=False, return_length=True)['length']
        print(source, 'Contributions:', len(texts), 'Longest tokens:', max(lengths), 'Limit:', model.max_seq_length)
        if max(lengths) > model.max_seq_length:
            raise ValueError('Contributions exceed model token limit. Review truncation before inference.')
    df['human_semantic'] = df['human_contributions'].apply(conversation_semantic_similarity)
    df['gpt_semantic'] = df['gpt_contributions'].apply(conversation_semantic_similarity)


In [ ]:
if RECOMPUTE_SEMANTIC:
    adjacent_results = []
    for source in ['human', 'gpt']:
        for _, row in df.iterrows():
            texts = [line.split(': ', 1)[1] for line in row[f'{source}_contributions'].splitlines()]
            embeddings = model.encode(texts)
            n_pairs = len(texts) - 1
            for i in range(1, len(texts)):
                adjacent_results.append({'pair_id': row['pair_id'], 'emotion': row['emotion'], 'source': source, 'transition': i, 'n_transitions': n_pairs, 'relative_position': (i - 1) / (n_pairs - 1) if n_pairs > 1 else 0.5, 'lexical': lexical_overlap(texts[i - 1], texts[i]), 'semantic': model.similarity(embeddings[i - 1:i], embeddings[i:i + 1]).item()})
    adjacent_df = pd.DataFrame(adjacent_results)
else:
    validate_cache()
    adjacent_df = pd.read_csv(OUTPUT_DIR / 'transition_scores.csv')
    print('Reusing supplied transition semantic scores. Embeddings were not rerun.')


In [ ]:
transition_columns = {'pair_id', 'emotion', 'source', 'transition', 'n_transitions', 'relative_position', 'lexical', 'semantic'}
if not transition_columns.issubset(adjacent_df.columns) or adjacent_df.duplicated(['pair_id', 'source', 'transition']).any():
    raise ValueError('Invalid transition schema or duplicate transitions')
expected_keys = {(pair_id, source) for pair_id in df.pair_id for source in ['human', 'gpt']}
if set(zip(adjacent_df.pair_id, adjacent_df.source)) != expected_keys or not np.isfinite(adjacent_df[['lexical', 'semantic']].to_numpy()).all():
    raise ValueError('Missing/extra dialogue keys or undefined transition scores')
if not adjacent_df.lexical.between(0, 1).all() or not adjacent_df.semantic.between(-1.000001, 1.000001).all():
    raise ValueError('Transition values outside metric ranges')
for _, row in df.iterrows():
    for source in ['human', 'gpt']:
        turns = parse_contributions(row[f'{source}_contributions'])
        part = adjacent_df.loc[adjacent_df.pair_id.eq(row.pair_id) & adjacent_df.source.eq(source)].sort_values('transition')
        n = len(turns) - 1
        positions = [(i - 1) / (n - 1) if n > 1 else 0.5 for i in range(1, n + 1)]
        if part.transition.tolist() != list(range(1, n + 1)) or not part.n_transitions.eq(n).all() or (not part.emotion.eq(row.emotion).all()) or (not np.allclose(part.relative_position, positions, atol=1e-12, rtol=0)):
            raise ValueError(f'Invalid transition structure: {row.pair_id}, {source}')
        lexical = [lexical_overlap(turns[i - 1][1], turns[i][1]) for i in range(1, len(turns))]
        if not np.allclose(part.lexical, lexical, atol=1e-12, rtol=0):
            raise ValueError(f'Lexical cache/extraction differs from input: {row.pair_id}, {source}')
recomputed = adjacent_df.groupby(['pair_id', 'source'])[['lexical', 'semantic']].mean().unstack('source')
for source in ['human', 'gpt']:
    if RECOMPUTE_SEMANTIC:
        original = df.set_index('pair_id')[f'{source}_semantic']
        if not np.allclose(recomputed['semantic', source].reindex(original.index), original, atol=1e-06, rtol=0):
            raise ValueError('Independent semantic aggregations disagree')
    df[f'{source}_semantic'] = df.pair_id.map(recomputed['semantic', source])
    if not np.allclose(df.pair_id.map(recomputed['lexical', source]), df[f'{source}_overlap'], atol=1e-12, rtol=0):
        raise ValueError('Lexical aggregations disagree')
df['semantic_difference'] = df['gpt_semantic'] - df['human_semantic']
print(df[['human_overlap', 'gpt_overlap', 'overlap_difference', 'human_semantic', 'gpt_semantic', 'semantic_difference']].mean())


## 3. Paired uncertainty and source distributions

Bootstrap 10,000 whole matched pairs, seed 42. Percentile confidence intervals describe uncertainty conditional on this curated sample. Resampling does not remove selection or length bias.

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)
bootstrap_results = []
for column in ['overlap_difference', 'semantic_difference']:
    differences = df[column].to_numpy()
    assert len(differences) == 320, 'Expected 320 matched pairs.'
    assert np.isfinite(differences).all(), 'Missing or invalid differences found.'
    bootstrap_means = rng.choice(differences, size=(10000, len(differences)), replace=True).mean(axis=1)
    lower, upper = np.percentile(bootstrap_means, [2.5, 97.5])
    bootstrap_results.append({'measure': column, 'mean_difference': differences.mean(), 'ci_lower': lower, 'ci_upper': upper})
confidence_intervals = pd.DataFrame(bootstrap_results)
print(confidence_intervals.round(4).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].boxplot([df['human_overlap'], df['gpt_overlap']], tick_labels=['Human', 'GPT'])
axes[0].set_title('Lexical overlap')
axes[0].set_ylabel('Mean adjacent Jaccard similarity')
axes[1].boxplot([df['human_semantic'], df['gpt_semantic']], tick_labels=['Human', 'GPT'])
axes[1].set_title('Semantic similarity')
axes[1].set_ylabel('Mean adjacent cosine similarity')
fig.suptitle('Conversational alignment — 320 matched pairs')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'alignment_distributions.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
lexical_gpt_higher = df['overlap_difference'] > 0
semantic_gpt_higher = df['semantic_difference'] > 0
agreement = pd.crosstab(lexical_gpt_higher, semantic_gpt_higher, rownames=['GPT higher lexically'], colnames=['GPT higher semantically'])
print(agreement)
print('Directional agreement:', f'{(lexical_gpt_higher == semantic_gpt_higher).mean():.1%}')
correlation = df[['overlap_difference', 'semantic_difference']].corr(method='spearman')
print(correlation)


## 4. First-three-transition sensitivity

Retain matched pairs with at least three transitions in both sources. This equalizes comparison counts and absolute turn positions, not utterance word length.

In [ ]:
transition_counts = adjacent_df.groupby(['pair_id', 'source']).size().unstack('source')
eligible_ids = transition_counts.index[(transition_counts['human'] >= 3) & (transition_counts['gpt'] >= 3)]
first_three = adjacent_df.loc[adjacent_df['pair_id'].isin(eligible_ids) & adjacent_df['transition'].between(1, 3)]
first_three_scores = first_three.groupby(['pair_id', 'source'])[['lexical', 'semantic']].mean().unstack('source')
print('Matched pairs retained:', len(first_three_scores))
for measure in ['lexical', 'semantic']:
    human = first_three_scores[measure, 'human']
    gpt = first_three_scores[measure, 'gpt']
    difference = gpt - human
    print(f'{measure}: Human={human.mean():.4f}, GPT={gpt.mean():.4f}, difference={difference.mean():.4f}, GPT higher={(difference > 0).sum()}/{len(difference)}')


## 5. Relative progression and consistency

Average transitions within each dialogue/stage, then average dialogues. Relative stages are unequal in absolute turn count between sources. Within-dialogue SD uses `ddof=1`.

In [ ]:
adjacent_df['stage'] = pd.cut(adjacent_df['relative_position'], bins=[-0.001, 1 / 3, 2 / 3, 1], labels=['Early', 'Middle', 'Late'])
dialogue_stages = adjacent_df.groupby(['pair_id', 'source', 'stage'], observed=True)[['lexical', 'semantic']].mean().reset_index()
progression = dialogue_stages.groupby(['source', 'stage'], observed=True)[['lexical', 'semantic']].mean()
print(progression)
print('\nNumber of dialogues contributing to each stage:')
print(dialogue_stages.groupby(['source', 'stage'], observed=True).size())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
stages = ['Early', 'Middle', 'Late']
for ax, measure, title in zip(axes, ['lexical', 'semantic'], ['Lexical overlap', 'Semantic similarity']):
    for source, label in [('human', 'Human'), ('gpt', 'GPT')]:
        values = progression.loc[source, measure].reindex(stages)
        ax.plot(stages, values, marker='o', label=label)
    ax.set_title(title)
    ax.set_xlabel('Relative conversation stage')
    ax.set_ylabel('Mean dialogue-stage score')
    ax.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'alignment_progression.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
dialogue_variability = adjacent_df.groupby(['pair_id', 'source'])[['lexical', 'semantic']].std().rename(columns={'lexical': 'lexical_sd', 'semantic': 'semantic_sd'}).reset_index()
print(dialogue_variability.groupby('source')[['lexical_sd', 'semantic_sd']].mean())
print('\nMissing variability scores:')
print(dialogue_variability[['lexical_sd', 'semantic_sd']].isna().sum())


## 6. Export tables and provenance

Flat column headers make the first-three scores easier to reload. No input CSV is modified. The retained day3_reviewed_320_v2.csv records pair IDs, emotions and review notes.

In [ ]:
score_columns = ['pair_id', 'emotion', 'human_overlap', 'gpt_overlap', 'overlap_difference', 'human_semantic', 'gpt_semantic', 'semantic_difference']
df[score_columns].to_csv(OUTPUT_DIR / 'dialogue_scores.csv', index=False)
adjacent_df.to_csv(OUTPUT_DIR / 'transition_scores.csv', index=False)
progression.to_csv(OUTPUT_DIR / 'progression_summary.csv')
first_three_export = first_three_scores.copy()
first_three_export.columns = [f'{source}_{measure}' for measure, source in first_three_export.columns]
first_three_export.to_csv(OUTPUT_DIR / 'first_three_scores.csv')
confidence_intervals.to_csv(OUTPUT_DIR / 'bootstrap_intervals.csv', index=False)
metadata = {'input_sha256': hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest(), 'model_inputs_sha256': hashlib.sha256(MODEL_INPUT_FILE.read_bytes()).hexdigest(), 'semantic_inference_rerun': RECOMPUTE_SEMANTIC, 'model': MODEL_NAME, 'n_pairs': len(df), 'bootstrap_samples': 10000, 'bootstrap_seed': 42, 'known_typo_replacements': typo_count, 'tokenizer': "[a-z]+(?:'[a-z]+)*"}
if RECOMPUTE_SEMANTIC:
    metadata.update({'sentence_transformers_version': importlib.metadata.version('sentence-transformers'), 'max_sequence_length': model.max_seq_length})
metadata['cache_input_sha256_lf'] = file_fingerprint(INPUT_FILE)
metadata['cache_sha256_lf'] = file_fingerprint(CACHE_FILE)
metadata['cache_model_inputs_sha256_lf'] = file_fingerprint(MODEL_INPUT_FILE)
metadata['cache_text_correction'] = 'hit:9551_conv:19102: mitig ates -> mitigates'
metadata['inference_provenance'] = {'python_version': sys.version, 'packages': {package: importlib.metadata.version(package) for package in ['numpy', 'pandas', "sentence-transformers", "transformers", "torch"]}, 'model_revision': 'Not recorded'} if RECOMPUTE_SEMANTIC else previous_metadata.get('inference_provenance', {'status': 'Historical inference details unavailable'})
(OUTPUT_DIR / 'alignment_run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved results to:', OUTPUT_DIR)
